# Notebook 06 — Community Detection in Networks

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 06 of 12  
**Time estimate:** 75 minutes

> Proteins cluster into functional modules. Genes co-express in pathways.
> Identifying these communities from network structure alone — without prior biological
> knowledge — is one of the most powerful tools in systems biology.

---
## Step 1 — Motivation

A PPI network with 5000 proteins and 100,000 interactions is too complex to interpret
directly. Community detection algorithms partition the network into dense subgraphs
(communities) that correspond — with striking frequency — to biological pathways,
protein complexes, and functional modules.

---
## Step 2 — Intuition

A **community** is a set of nodes that are densely connected internally and sparsely
connected to the rest of the network.

**Modularity Q** measures how much the actual edge density within communities
exceeds what would be expected in a random graph with the same degree sequence.
Maximizing Q is the standard community detection objective.

**Louvain algorithm** (Blondel et al. 2008):
1. Start with each node in its own community
2. For each node, try moving it to each neighbor's community — keep if ΔQ > 0
3. Repeat sweeps until no improvement
4. Aggregate: each community becomes a super-node; repeat

Greedy modularity maximization, $O(n \log n)$ in practice.

**Leiden algorithm** (Traag et al. 2019) improves Louvain by guaranteeing
well-connected communities — Louvain can produce internally disconnected communities.

---
## Step 3 — Biological Background

**Functional modules:** Communities in PPI networks are enriched for specific GO terms
(Gene Ontology annotations) — proteins that work together physically tend to cluster
together structurally.

**Disease modules (Menche et al. 2015):** Genes associated with the same disease
form a connected subnetwork (module) in the PPI. Disease modules for similar diseases
overlap; modules for unrelated diseases don't. This is the "disease module hypothesis."

**Co-expression modules (WGCNA):** In transcriptomics, genes with correlated
expression across conditions are clustered using hierarchical clustering on the
correlation matrix, creating a weighted gene network. The modules correlate with
traits (cell types, disease states).

**Planted partition model (benchmark):** Generate a ground-truth modular network
by creating cliques and adding random inter-module edges. Test community detection
by how well it recovers the planted partition.
- Stochastic Block Model (SBM): $P(\text{edge between } i \in C_a, j \in C_b) = p_{ab}$
- When $p_{aa} \gg p_{ab}$, communities are detectable

---
## Step 4 — Mathematical Explanation

**Modularity:**
$$Q = \frac{1}{2m} \sum_{ij} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j)$$

where $m = |E|$, $A_{ij}$ is the adjacency matrix, $k_i k_j / 2m$ is the expected
number of edges between $i$ and $j$ in a random null model (same degrees, random
connections), and $\delta(c_i, c_j) = 1$ if $i,j$ are in the same community.

$Q \in [-0.5, 1]$; good partitions typically have $Q > 0.3$.

**Modularity gain from moving node $v$ to community $C$:**
$$\Delta Q = \left[ \frac{k_{v,\text{in}}}{m} - \frac{k_v \Sigma_{\text{tot}}}{2m^2} \right]$$

where $k_{v,\text{in}}$ = edges from $v$ to nodes in $C$, $\Sigma_{\text{tot}}$ = sum of degrees in $C$.
This can be computed in $O(\deg(v))$ per node per iteration.

**Normalized Mutual Information (NMI):** Quality measure comparing detected
partition to ground truth:
$$\text{NMI}(A, B) = \frac{2 I(A; B)}{H(A) + H(B)} \in [0, 1]$$

**Adjusted Rand Index (ARI):** $\text{ARI} \in [-1, 1]$; 1 = perfect agreement;
0 = random; negative = worse than random.

In [ ]:
# Step 6 — Python: Stochastic Block Model + Louvain via NetworkX
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict

# ---- Generate Stochastic Block Model graph (planted partition) ----
def stochastic_block_model(block_sizes, p_in, p_out, seed=42):
    """
    Planted partition graph.
    block_sizes : list of ints, number of nodes in each block
    p_in  : float, within-block edge probability
    p_out : float, between-block edge probability
    Returns: adj (dict), true_labels (list)
    """
    rng = np.random.default_rng(seed)
    n = sum(block_sizes)
    adj = {i: set() for i in range(n)}
    true_labels = []
    for block_id, size in enumerate(block_sizes):
        true_labels += [block_id] * size
    true_labels = np.array(true_labels)
    for i in range(n):
        for j in range(i+1, n):
            p = p_in if true_labels[i] == true_labels[j] else p_out
            if rng.random() < p:
                adj[i].add(j)
                adj[j].add(i)
    return adj, true_labels


# Create a 4-module network: 4 blocks of 50 nodes each
adj_sbm, true_labels = stochastic_block_model(
    block_sizes=[50, 50, 50, 50],
    p_in=0.2, p_out=0.02, seed=42
)

# Build NetworkX graph for Louvain
G = nx.Graph()
for v in adj_sbm:
    for u in adj_sbm[v]:
        if u > v:
            G.add_edge(v, u)

# ---- Louvain community detection ----
try:
    import community as community_louvain
    partition = community_louvain.best_partition(G, random_state=42)
    detected_labels = np.array([partition[v] for v in sorted(G.nodes())])
    method = 'Louvain (python-louvain)'
except ImportError:
    try:
        from networkx.algorithms.community import louvain_communities
        comms = louvain_communities(G, seed=42)
        detected_labels = np.zeros(len(adj_sbm), dtype=int)
        for cid, comm in enumerate(comms):
            for node in comm:
                detected_labels[node] = cid
        method = 'Louvain (NetworkX)'
    except Exception:
        # Fallback: greedy modularity maximization
        comms = nx.algorithms.community.greedy_modularity_communities(G)
        detected_labels = np.zeros(len(adj_sbm), dtype=int)
        for cid, comm in enumerate(comms):
            for node in comm:
                detected_labels[node] = cid
        method = 'Greedy modularity (fallback)'

print(f'Community detection method: {method}')
n_detected = len(np.unique(detected_labels))
print(f'Detected communities: {n_detected} (planted: 4)')

# ---- Compute modularity Q from scratch ----
def modularity(adj, labels):
    nodes = sorted(adj.keys())
    m = sum(len(adj[v]) for v in nodes) / 2
    if m == 0: return 0.0
    k = {v: len(adj[v]) for v in nodes}
    Q = 0.0
    for v in nodes:
        for u in adj[v]:
            if labels[v] == labels[u]:
                Q += 1 - k[v] * k[u] / (2*m)
    return Q / (2*m)

Q_detected = modularity(adj_sbm, detected_labels)
print(f'Modularity Q (detected): {Q_detected:.4f}')

# ---- ARI ----
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(true_labels, detected_labels)
print(f'Adjusted Rand Index (vs ground truth): {ari:.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Layout: block-structured circular for clarity
nodes = sorted(adj_sbm.keys())
theta = np.linspace(0, 2*np.pi, len(nodes), endpoint=False)
# Group by true block to make layout show structure
block_order = np.argsort(true_labels * 1000 + np.arange(len(nodes)))
pos = np.column_stack([np.cos(theta[block_order.argsort()]),
                         np.sin(theta[block_order.argsort()])])

cmap_4 = plt.cm.get_cmap('tab10', 4)
cmap_det = plt.cm.get_cmap('tab10', n_detected)

# Panel A: Ground truth
ax = axes[0]
for u in adj_sbm:
    for v in adj_sbm[u]:
        if u < v:
            ax.plot([pos[u,0], pos[v,0]], [pos[u,1], pos[v,1]],
                      'grey', lw=0.15, alpha=0.3, zorder=1)
ax.scatter(pos[:,0], pos[:,1], c=[cmap_4(true_labels[v]) for v in nodes],
             s=25, zorder=2)
ax.set_title(f'A. Ground truth (4 communities)', fontsize=8)
ax.axis('off')

# Panel B: Detected communities
ax = axes[1]
for u in adj_sbm:
    for v in adj_sbm[u]:
        if u < v:
            ax.plot([pos[u,0], pos[v,0]], [pos[u,1], pos[v,1]],
                      'grey', lw=0.15, alpha=0.3, zorder=1)
ax.scatter(pos[:,0], pos[:,1], c=[cmap_det(detected_labels[v]) for v in nodes],
             s=25, zorder=2)
ax.set_title(f'B. Detected ({n_detected} communities)\nQ={Q_detected:.3f}, ARI={ari:.3f}', fontsize=8)
ax.axis('off')

# Panel C: Confusion matrix (ground truth vs detected)
ax = axes[2]
confusion = np.zeros((4, n_detected), dtype=int)
for v in nodes:
    confusion[true_labels[v], detected_labels[v]] += 1
im = ax.imshow(confusion, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Detected community')
ax.set_ylabel('True block')
ax.set_title('C. Confusion matrix\n(rows=true, cols=detected)')
ax.set_xticks(range(n_detected))
ax.set_yticks(range(4))

plt.suptitle('Module 12 NB06: Community Detection (Louvain / Greedy Modularity)',
               fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('community_detection.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Reduce `p_out` to 0.005. How does ARI change? What about Q? (*Stronger module signal.*)
2. Increase `p_out` to 0.08 (approaching `p_in`). At what ratio `p_out/p_in` do
   communities become undetectable?
3. Implement modularity computation $Q$ from scratch. Verify it matches for the
   detected partition above.
4. What biological experiment would you run to validate that a detected network
   community corresponds to a real functional module?

---
## Step 10 — Quiz

1. What does modularity $Q$ measure? What values indicate a good partition?
2. Describe the two phases of the Louvain algorithm.
3. What is the planted partition / Stochastic Block Model?
4. What is ARI? Why is it preferred over raw accuracy for clustering evaluation?
5. What biological entities do network communities often correspond to?

---
## Step 12 — Reflection

> *[In one paragraph: explain the relationship between detected network communities
> and biological pathways. What would it mean if a detected community contains
> proteins from two known pathways?]*

---
*Next: `07_ppi_network_analysis.ipynb`*